# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Agustín Accurso    
- Alumno 2 : Lautaro Cena

## 1. Introducción y descripción del sistema

Este trabajo práctico implementa un sistema completo de reconocimiento e 
identificación facial, desarrollado en el marco de la materia IA 5.2 Computer Vision.

El sistema es capaz de detectar rostros en imágenes, extraer representaciones 
vectoriales (embeddings), identificar personas registradas y manejar casos de 
desconocidos.

El pipeline sigue el enfoque estándar de sistemas modernos:

    Detección → Alineación → Embeddings → Comparación

La arquitectura fue provista por la cátedra e incluye:
- Backend: FastAPI con procesamiento asincrónico
- Frontend: Gradio
- Base de datos vectorial: PostgreSQL + pgvector
- Modelo: InsightFace (buffalo_l) con ArcFace para extracción de embeddings

Se implementaron las funciones detect_faces, align_face y 
extract_embedding_from_face dentro del servicio de procesamiento facial.

# 2. Dataset

El dataset fue construido con fotos propias variando las siguientes condiciones:
- Iluminación: natural, artificial, baja luz
- Pose: frente, leve perfil, ángulo desde abajo (selfie)
- Contexto: interior, exterior, espejo
- Vestimenta: distintos looks para evitar que el modelo se apoye en ropa

Las fotos grupales fueron recortadas manualmente para garantizar una sola cara 
por imagen, requisito del endpoint /insert.

No se utilizaron datasets públicos dado que el objetivo del sistema es el 
reconocimiento de personas conocidas con embeddings preentrenados, donde 
no se requiere reentrenamiento del modelo.

| Persona | Cantidad de imágenes | Condiciones | Fuente |
|---------|---------------------|-------------|--------|
| Agus    | 12                  | Selfies, fotos sociales, distintas luces, indoor/outdoor | Fotos personales |º
| Silvia    | 13                  | Selfies, fotos sociales, distintas luces, indoor/outdoor | Fotos personales |º

In [4]:
from pathlib import Path

dataset_path = Path("data")
personas = [p for p in dataset_path.iterdir() if p.is_dir()]

total = 0
for persona in personas:
    imagenes = list(persona.glob("*.jpg")) + list(persona.glob("*.png"))
    print(f"{persona.name}: {len(imagenes)} imágenes")
    total += len(imagenes)

print(f"\nTotal personas: {len(personas)}")
print(f"Total imágenes: {total}")

agus: 12 imágenes
silvia: 13 imágenes

Total personas: 2
Total imágenes: 25


# 3. Preprocesamiento

El preprocesamiento es realizado internamente por InsightFace como parte 
del pipeline de detección y alineación:

- Resize: las caras son normalizadas a 112x112 píxeles (FACE_SIZE=112)
- Normalización: media y desviación estándar aplicadas internamente por ArcFace
- Alineación geométrica: norm_crop transforma la cara usando los 5 keypoints 
  (ojos, nariz, comisuras) para centrar y enderezar el rostro
- No se aplica data augmentation ya que se usan embeddings preentrenados,
  no se entrena el modelo desde cero

Filtrado de calidad: el sistema rechaza imágenes donde no se detecta 
exactamente una cara (register_identity lanza ValueError si hay 0 o más de 1).

# 4. Modelo (Backbone)


### Modelo elegido: InsightFace — buffalo_l (ArcFace + RetinaFace)

Se utilizó el modelo preentrenado buffalo_l de InsightFace, que integra
dos componentes:

- **Detección**: RetinaFace — detecta caras y extrae 5 keypoints faciales
- **Embeddings**: ArcFace (ResNet50) — genera embeddings de 512 dimensiones

### Justificación de la elección

Se optó por un modelo preentrenado sin fine-tuning por las siguientes razones:

1. El objetivo es identificación de personas conocidas, no clasificación 
   de un dataset cerrado. Los embeddings preentrenados generalizan bien 
   a caras no vistas durante el entrenamiento.

2. El hardware disponible es CPU (Ryzen 5 7430U) sin GPU dedicada. 
   Entrenar o hacer fine-tuning desde cero es inviable en este contexto.

3. ArcFace fue entrenado con millones de caras (MS1MV2, ~5.8M imágenes), 
   lo que garantiza representaciones robustas sin necesidad de reentrenamiento.

### Trade-offs

| Criterio             | buffalo_l         | buffalo_sc        |
|----------------------|-------------------|-------------------|
| Precisión            | Alta              | Media             |
| Velocidad en CPU     | Más lenta         | Más rápida        |
| Tamaño del modelo    | ~300MB            | ~30MB             |
| Uso en este TP       | ✓ Elegido         |                   |

Se priorizó precisión sobre velocidad dado que el sistema procesa 
imágenes estáticas, no video en tiempo real.

### Arquitectura interna de ArcFace (ResNet50)

- Input: imagen alineada 112x112 px, 3 canales BGR
- Backbone: ResNet50 con capas convolucionales profundas
- Output: vector de 512 dimensiones (L2-normalizado)
- Loss de entrenamiento: ArcFace Loss (variante de Softmax con margen angular)
  que maximiza la separabilidad entre clases en el espacio esférico

# 5. Embeddings

### Representación vectorial

Cada cara registrada se representa como un vector de 512 dimensiones 
generado por ArcFace. Estos vectores capturan características faciales 
abstractas aprendidas durante el entrenamiento.

### Almacenamiento

Se utiliza PostgreSQL + pgvector (provisto por la cátedra) como base 
vectorial. Cada registro tiene la siguiente estructura:

- id_imagen: identificador único (UUID)
- embedding: vector de 512 floats
- etiqueta: nombre de la persona
- path: ruta de la imagen de referencia guardada
- metadata: información adicional (extensible)

### Búsqueda por similitud

Para identificar una cara, se compara su embedding contra todos los 
registrados usando similitud coseno:

    similitud(a, b) = (a · b) / (||a|| * ||b||)

El resultado es un score entre 0 y 1. Si el mejor score supera el 
threshold (SIMILARITY_THRESHOLD=0.55), se retorna la etiqueta 
correspondiente. Caso contrario se retorna "unknown".

# 6. Evaluación y métricas

# 7. Conclusiones

## Problemas encontrados y soluciones

* Las coordenadas son correctas en el espacio de la imagen original, el problema es de renderizado en el frontend que no fue modificado.